In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import multiprocessing
from functools import partial

File_Path_str = os.path.expanduser('~/DJAv4')
DJA_Root_Url_str = "https://s3.amazonaws.com/msaexp-nirspec/extractions"
os.makedirs(File_Path_str, exist_ok=True)
DJA_v4_Catalog_Path_str = './DJAv4Catalog.csv'
DJA_v4_Catalog_DataFrame = pd.read_csv(DJA_v4_Catalog_Path_str)

def Download_FITS(Index_In_DJA_v4_DataFrame_int, DJA_v4_DataFrame, DJA_Root_Url_str, File_Path_str):
    """
    Download FITS files from the DJA v4 catalog provided.
    Parameters
    ----------
    Index_In_DJA_v4_DataFrame_int : int
        Index of the row in the DataFrame to download.
    DJA_v4_DataFrame : pd.DataFrame
        DataFrame containing the DJA v4 catalog.
    DJA_Root_Url_str : str
        Root URL for the DJA v4 catalog.
    File_Path_str : str
        Path to save the downloaded FITS files.
    Returns
    -------
    int
        0 for success, -1 for failure
    """
    # Get the row from the DataFrame
    try:
        Object_Catalog = DJA_v4_DataFrame.iloc[Index_In_DJA_v4_DataFrame_int]
        Object_Root_str = Object_Catalog.root
        Object_FileName_str = Object_Catalog.file

        # Create the full file path
        Fits_File_Root_Path_str = os.path.join(File_Path_str, Object_Root_str)
        Fits_File_Full_Path_str = os.path.join(Fits_File_Root_Path_str, Object_FileName_str)
        Fits_File_Url_str = f"{DJA_Root_Url_str}/{Object_Root_str}/{Object_FileName_str}"

        # Create directory if it doesn't exist
        os.makedirs(Fits_File_Root_Path_str, exist_ok=True)

        # Check if file already exists
        if os.path.exists(Fits_File_Full_Path_str):
            return 0

        # Download the FITS file
        result = os.system(f'wget -q -P {Fits_File_Root_Path_str} {Fits_File_Url_str}')

        if result == 0:
            return 0
        else:
            print(f"wget failed with return code: {result}")
            return -1

    except Exception as e:
        print(f"Error downloading file at index {Index_In_DJA_v4_DataFrame_int}: {e}")
        return -1

def main():
    File_Path_str = os.path.expanduser('~/DJAv4')
    DJA_Root_Url_str = "https://s3.amazonaws.com/msaexp-nirspec/extractions"
    DJA_v4_Catalog_Path_str = './DJAv4Catalog.csv'
    DJA_v4_Catalog_DataFrame = pd.read_csv(DJA_v4_Catalog_Path_str)
    DJA_v4_Catalog_DataFrame = DJA_v4_Catalog_DataFrame.sort_values(by='root')

    print(f"\n{'='*60}")
    print(f"Downloading data")
    print(f"{'='*60}\n")

    total_files_int = len(DJA_v4_Catalog_DataFrame)
    print(f"Total files to download: {total_files_int}")

    # Use multiprocessing to download files in parallel
    num_processes_int = int(multiprocessing.cpu_count() * 0.8)
    print(f"Using {num_processes_int} processes for parallel downloading.")

    pool = multiprocessing.Pool(processes=num_processes_int)

    # Create partial function with the fixed parameters
    download_func = partial(Download_FITS,
                           DJA_v4_DataFrame=DJA_v4_Catalog_DataFrame,
                           DJA_Root_Url_str=DJA_Root_Url_str,
                           File_Path_str=File_Path_str)

    results = list(tqdm(pool.imap(download_func, range(total_files_int)),
                       total=total_files_int,
                       desc="Downloading FITS files"))

    pool.close()
    pool.join()

    successful_downloads = sum(1 for result in results if result == 0)
    failed_downloads = sum(1 for result in results if result == -1)

    print(f"\n{'='*60}")
    print(f"Download completed.")
    print(f"Total files downloaded successfully: {successful_downloads}")
    print(f"Total files failed to download: {failed_downloads}")
    print(f"Total files in catalog: {total_files_int}")
    print(f"Download rate: {successful_downloads/total_files_int*100:.2f}%")
    print(f"Failed rate: {failed_downloads/total_files_int*100:.2f}%")
    print(f"{'='*60}\n")

if __name__ == "__main__":
    main()



Total files to download: 67099
Using 8 processes for parallel downloading.


Traceback (most recent call last):
  File "/opt/anaconda3/envs/dustcurve/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/envs/dustcurve/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/anaconda3/envs/dustcurve/lib/python3.11/multiprocessing/pool.py", line 114, in worker
    task = get()
           ^^^^^
  File "/opt/anaconda3/envs/dustcurve/lib/python3.11/multiprocessing/queues.py", line 367, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'Download_FITS' on <module '__main__' (built-in)>
Process SpawnPoolWorker-2:
Traceback (most recent call last):
  File "/opt/anaconda3/envs/dustcurve/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/envs/dustcurve/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._

KeyboardInterrupt: 

Process SpawnPoolWorker-326:
Traceback (most recent call last):
  File "/opt/anaconda3/envs/dustcurve/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/envs/dustcurve/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/anaconda3/envs/dustcurve/lib/python3.11/multiprocessing/pool.py", line 114, in worker
    task = get()
           ^^^^^
  File "/opt/anaconda3/envs/dustcurve/lib/python3.11/multiprocessing/queues.py", line 367, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'Download_FITS' on <module '__main__' (built-in)>


Process SpawnPoolWorker-327:
Traceback (most recent call last):
  File "/opt/anaconda3/envs/dustcurve/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/envs/dustcurve/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/anaconda3/envs/dustcurve/lib/python3.11/multiprocessing/pool.py", line 114, in worker
    task = get()
           ^^^^^
  File "/opt/anaconda3/envs/dustcurve/lib/python3.11/multiprocessing/queues.py", line 367, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'Download_FITS' on <module '__main__' (built-in)>
Process SpawnPoolWorker-328:
Traceback (most recent call last):
  File "/opt/anaconda3/envs/dustcurve/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/envs/dustcurve/lib/python3.11/multiprocessing/process.py", line 108, 

In [3]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import multiprocessing
from functools import partial

File_Path_str = os.path.expanduser('~/DJAv4')
DJA_Root_Url_str = "https://s3.amazonaws.com/msaexp-nirspec/extractions"
os.makedirs(File_Path_str, exist_ok=True)
DJA_v4_Catalog_Path_str = './DJAv4Catalog.csv'
DJA_v4_Catalog_DataFrame = pd.read_csv(DJA_v4_Catalog_Path_str)
DJA_v4_Catalog_Path_str = './DJAv4Catalog.csv'
DJA_v4_Catalog_DataFrame = pd.read_csv(DJA_v4_Catalog_Path_str)
DJA_v4_Catalog_DataFrame = DJA_v4_Catalog_DataFrame.sort_values(by='root')